# 🤖 **AutoML usando PyCaret**

En este notebook vamos a usar PyCaret para entrenar y comparar múltiples modelos de clasificación de manera automática, con el fin de identificar el modelo con mejor rendimiento para la predicción de la variable `default`.

---

## **Librerias**

In [49]:
from pathlib import Path
import numpy as np
import pandas as pd
from pycaret.classification import setup, compare_models
from pycaret.classification import create_model, tune_model, predict_model
from pycaret.classification import save_model
from pycaret.classification import load_model
import joblib

In [2]:
DATA_DIR = Path.cwd().parent / "data" / "processed"
datos_cooperativa_ajustados = pd.read_parquet(DATA_DIR / "02_datos_ajustados_cooperativa.parquet")

## ➜ **Selección de Columnas**

Después de realizar el análisis exploratorio de datos (EDA) y determinar qué variables no serán utilizadas, se seleccionaron las siguientes variables para el entrenamiento del modelo:

In [3]:
datos_cooperativa_ajustados.columns

Index(['cartera', 'plazo', 'vinculacion', 'valor_cuota', 'valor_prestamo',
       'saldo_capital', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'reestr', 'ctasahorros', 'edad', 'tipoasociado',
       'estado_cliente', 'departamento', 'sexo', 'curtotalingresos',
       'curtotalegresos', 'intestrato', 'actualizacion', 'default',
       'puntaje_data', 'grupo_dptmto', 'grupo_ciudad', 'grupo_edad',
       'grupo_actividadeco'],
      dtype='object')

In [4]:
columnas_seleccionadas = [
    'cartera', 
    'plazo', 
    'vinculacion', 
    'valor_cuota', 
    'valor_prestamo',
    'saldo_capital', 
    'saldo_interes', 
    'aportes', 
    'garantias',
    'valorgarantia', 
    'ctasahorros', 
    'edad', 
    'tipoasociado',
    'sexo', 
    'curtotalingresos',
    'curtotalegresos', 
    'intestrato', 
    'actualizacion', 
    'default',
    'puntaje_data', 
    'grupo_ciudad', 
    'grupo_edad',
    'grupo_actividadeco']

In [5]:
df_cooperativa = datos_cooperativa_ajustados[columnas_seleccionadas]
df_cooperativa.head()

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data,grupo_ciudad,grupo_edad,grupo_actividadeco
0,consumo_sin_libranza,1827,8103,356849.0,15000000.0,12923538.0,123855.0,7741255,1,7741255.0,...,0,4597000.0,1500000.0,5,1,False,795.0,1,3,4
1,consumo_sin_libranza,1826,1434,2650409.0,100460000.0,31911361.0,263265.0,4601706,1,4601706.0,...,0,4597000.0,650000.0,5,1,False,836.0,5,3,4
2,consumo_sin_libranza,1826,573,791482.0,30000000.0,23844684.0,261477.0,530431,1,530431.0,...,0,4400000.0,2000000.0,4,0,True,709.0,7,2,4
3,consumo_sin_libranza,2922,1902,2860501.0,176000000.0,113842595.0,1008570.0,3023534,2,320385440.0,...,0,22020000.0,1500000.0,4,1,False,733.0,6,3,1
4,consumo_sin_libranza,2557,1902,987637.0,50300000.0,38521256.0,317167.0,1023082,2,320385440.0,...,0,22020000.0,1500000.0,4,1,False,695.0,6,3,1


In [6]:
df_cooperativa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12909 entries, 0 to 12908
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   cartera             12909 non-null  category
 1   plazo               12909 non-null  int64   
 2   vinculacion         12909 non-null  int64   
 3   valor_cuota         12909 non-null  float64 
 4   valor_prestamo      12909 non-null  float64 
 5   saldo_capital       12909 non-null  float64 
 6   saldo_interes       12909 non-null  float64 
 7   aportes             12909 non-null  int64   
 8   garantias           12909 non-null  int64   
 9   valorgarantia       12909 non-null  float64 
 10  ctasahorros         12909 non-null  float64 
 11  edad                12909 non-null  int32   
 12  tipoasociado        12909 non-null  int64   
 13  sexo                12909 non-null  int64   
 14  curtotalingresos    12909 non-null  float64 
 15  curtotalegresos     12909 non-null  

In [13]:
columnas_numericas = df_cooperativa.select_dtypes(include=['int64', 'int32', 'float64']).columns
print("Columnas numéricas:", columnas_numericas)
print("\nNúmero de columnas numéricas:", len(columnas_numericas))

columnas_categoricas = df_cooperativa.select_dtypes(include=['category']).columns
print("Columnas categóricas:", columnas_categoricas)
print("\nNúmero de columnas categóricas:", len(columnas_categoricas))


Columnas numéricas: Index(['plazo', 'vinculacion', 'valor_cuota', 'valor_prestamo',
       'saldo_capital', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'ctasahorros', 'edad', 'tipoasociado', 'sexo',
       'curtotalingresos', 'curtotalegresos', 'intestrato', 'actualizacion',
       'puntaje_data', 'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco'],
      dtype='object')

Número de columnas numéricas: 21
Columnas categóricas: Index(['cartera'], dtype='object')

Número de columnas categóricas: 1


## ➜ **AutoML**

Definimos nuestra variable objetivo

In [7]:
target_col = "default"

In [15]:
cols_categoricas = ['cartera']

cols_numericas = ['plazo', 'vinculacion', 'valor_cuota', 'valor_prestamo',
       'saldo_capital', 'saldo_interes', 'aportes', 'garantias',
       'valorgarantia', 'ctasahorros', 'edad', 'tipoasociado', 'sexo',
       'curtotalingresos', 'curtotalegresos', 'intestrato', 'actualizacion',
       'puntaje_data', 'grupo_ciudad', 'grupo_edad', 'grupo_actividadeco']


In [16]:
## Llamamos el setup de PyCaret para configurar el entorno de entrenamiento del modelo
clf_setup = setup(
    data=df_cooperativa,
    target=target_col,
    session_id=42,

    numeric_features=cols_numericas,
    categorical_features=cols_categoricas,

    imputation_type=None
)

,Description,Value
0,Session id,42
1,Target,default
2,Target type,Binary
3,Original data shape,"(12909, 23)"
4,Transformed data shape,"(12909, 25)"
5,Transformed train set shape,"(9036, 25)"
6,Transformed test set shape,"(3873, 25)"
7,Numeric features,21
8,Categorical features,1
9,Preprocess,True


En este caso no fue necesario aplicar técnicas de imputación, ya que el conjunto de datos no presenta valores nulos. 

Por esta razón, se desactivó la imputación automática en PyCaret utilizando `imputation_type=None`, permitiendo trabajar directamente con los datos previamente limpios y evitando transformaciones innecesarias durante el preprocesamiento.


## ➜ **Comparación de varios modelos**

In [21]:
best_model = compare_models(
    include=["lr", "rf", "gbc", "lightgbm"],
    sort="Recall",
    n_select=4
)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9334,0.9609,0.7481,0.8500,0.7953,0.7557,0.7583,0.2830
gbc,Gradient Boosting Classifier,0.9283,0.9581,0.7059,0.8556,0.7729,0.7308,0.7360,1.0150
rf,Random Forest Classifier,0.9284,0.9568,0.6880,0.8716,0.7685,0.7269,0.7343,0.4890
lr,Logistic Regression,0.8380,0.8083,0.1055,0.7236,0.1833,0.1457,0.2327,0.5920


En este proyecto se utilizará la métrica **Recall** como criterio principal para comparar los modelos, debido a que el conjunto de datos se encuentra desbalanceado. 

En este tipo de escenarios, la métrica Accuracy puede resultar engañosa, ya que un modelo podría obtener una alta exactitud simplemente prediciendo la clase mayoritaria.

El Recall permite evaluar qué tan bien el modelo identifica los casos positivos, es decir, los clientes que realmente presentan riesgo de impago. Esta métrica es especialmente importante en problemas de riesgo crediticio, porque un falso negativo representa un caso en el que el modelo clasifica como confiable a un cliente que realmente podría incumplir sus pagos.

Por esta razón, se prioriza maximizar el Recall, buscando detectar la mayor cantidad posible de clientes riesgosos.


---

Los resultados muestran que el modelo con mejor desempeño en términos de **Recall** es **Light Gradient Boosting Machine (LightGBM)** con un valor de **0.7481**, seguido por **Gradient Boosting Classifier** con **0.7059**. 

Esto indica que LightGBM logra identificar más o menos el 74.81% de los clientes con riesgo de impago, siendo el modelo más efectivo para detectar la clase positiva dentro del conjunto de datos desbalanceado.

Aunque otros modelos como Random Forest presentan una mayor precisión, su Recall es menor. Por otro lado, Logistic Regression obtuvo un Recall bastante bajo (**0.1055**), evidenciando dificultades para identificar correctamente la clase minoritaria.

Debido a que en este problema es más importante detectar clientes con posible incumplimiento que maximizar únicamente la exactitud general, se selecciona **LightGBM** como el modelo con mejor rendimiento, ya que ofrece el mayor Recall.


In [ ]:
# 1. Crear el modelo base LightGBM
lightgbm_model = create_model("lightgbm")

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9381,0.9764,0.7885,0.8425,0.8146,0.7774,0.7780
1,0.9204,0.9449,0.7244,0.7958,0.7584,0.7108,0.7120
2,0.9369,0.9602,0.7643,0.8571,0.8081,0.7705,0.7723
3,0.9248,0.9660,0.7643,0.7947,0.7792,0.7339,0.7341
4,0.9259,0.9446,0.7006,0.8462,0.7666,0.7230,0.7275
5,0.9381,0.9557,0.7452,0.8797,0.8069,0.7703,0.7741
6,0.9369,0.9700,0.7500,0.8667,0.8041,0.7667,0.7696
7,0.9457,0.9723,0.7949,0.8794,0.8350,0.8026,0.8041
8,0.9369,0.9635,0.7436,0.8722,0.8028,0.7655,0.7689


El modelo **LightGBM** tiene un buen desempeño en general. Su accuracy es de aproximadamente 0.93 y su AUC es de 0.96, lo que significa que es bueno para distinguir entre las clases.

Además, la baja desviación estándar muestra que el modelo es estable, es decir, se comporta de forma similar en diferentes pruebas sin dispersión.

En general, es un modelo confiable y con buen rendimiento, pero todavía puede mejorar en detectar mejor la clase positiva ya que el recall no es perfecto y tiene un valor de 0.74 aproximadamente, lo que significa que el modelo aún no identifica todos los casos positivos reales.



In [24]:
# 2. Afinar hiperparámetros
print("Fine-tuning del modelo LightGBM")

lightgbm_tuned = tune_model(lightgbm_model)

Fine-tuning del modelo LightGBM


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9425,0.9771,0.8077,0.8514,0.8289,0.7944,0.7948
1,0.9237,0.9372,0.7308,0.8085,0.7677,0.7222,0.7235
2,0.9447,0.9634,0.8089,0.8639,0.8355,0.8023,0.8030
3,0.9392,0.9676,0.7771,0.8592,0.8161,0.7797,0.7811
4,0.9248,0.9458,0.7197,0.8248,0.7687,0.7240,0.7264
5,0.9270,0.9560,0.7325,0.8273,0.7770,0.7336,0.7355
6,0.9247,0.9649,0.7115,0.8284,0.7655,0.7210,0.7239
7,0.9468,0.9706,0.7949,0.8857,0.8378,0.8062,0.8078
8,0.9358,0.9652,0.7372,0.8712,0.7986,0.7607,0.7645


Fitting 10 folds for each of 10 candidates, totalling 100 fits


Después del ajuste de hiperparámetros con LightGBM, el modelo mantiene un desempeño muy similar al inicial. 

El accuracy y el AUC siguen siendo altos, lo que indica que el poder general de clasificación no cambió significativamente. 

El recall también se mantiene alrededor de 0.75, por lo que la capacidad de detectar la clase positiva no tuvo una mejora tan significativa.

En general, el tuning no cambió drásticamente el rendimiento del modelo, pero ayudó a afinar su estabilidad y balance entre métricas.



* No mejoró mucho en métricas principales
* Se mantuvo igual de bueno
* Se volvió un poco más estable
* No hubo un “salto” grande de rendimiento

In [25]:
print("Resumen del modelo ajustado:")

print(lightgbm_tuned)

Resumen del modelo ajustado:
LGBMClassifier(bagging_fraction=0.8, bagging_freq=3, boosting_type='gbdt',
               class_weight=None, colsample_bytree=1.0, feature_fraction=0.8,
               importance_type='split', learning_rate=0.2, max_depth=-1,
               min_child_samples=6, min_child_weight=0.001, min_split_gain=0.6,
               n_estimators=100, n_jobs=-1, num_leaves=30, objective=None,
               random_state=42, reg_alpha=0.001, reg_lambda=5, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)


Para el modelo se realizaron los siguientes ajustes:

* Submuestreo de variables (`feature_fraction=0.8`) y de datos (`bagging_fraction=0.8`, `bagging_freq=3`) para reducir el sobreajuste

* Ajuste de parámetros de regularización como `reg_alpha`, `reg_lambda` y `min_split_gain` que ayudan a controlar la complejidad del modelo.

* Ajuste de parámetros de estructura como `num_leaves` y `max_depth` para equilibrar la capacidad de aprendizaje con la generalización. 

En general, este conjunto de configuraciones busca que el modelo mantenga un buen desempeño sin memorizar los datos.


In [26]:
print("Generando predicciones en el conjunto de validación interno:")

predicciones_internas = predict_model(lightgbm_tuned)

Generando predicciones en el conjunto de validación interno:


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.9334,0.9644,0.7597,0.8399,0.7978,0.7580,0.7594


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


En la parte de las predicciones:

* El modelo acierta la mayoría de los casos en general ya que presenta un accuracy de 0.93.

* Tiene buena capacidad para distinguir clases con un valor de 0.96.

* Detecta el 0.76 de los casos positivos reales aunque aún se le siguen escapando algunos.

* La mayoria de ñas veces que predice positivo, es correcto ya que hay un precisión de 0.84.

* Analizando F1, se presenta un buen equilibrio entre precision y recall

**CONCLUSIÓN GENERAL**

El modelo **LightGBM** es estable, consistente y tiene buen desempeño en datos no vistos, sin señales claras de sobreajuste.

In [29]:
# Mostrar primeras predicciones
predicciones_internas.head()

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,curtotalegresos,intestrato,actualizacion,puntaje_data,grupo_ciudad,grupo_edad,grupo_actividadeco,default,prediction_label,prediction_score
981,consumo_sin_libranza,915,2340,355423.0,9040000.0,5175913.0,28470.0,4960189,1,4960189.0,...,800000.0,4,0,773.0,7,2,4,False,0,0.9489
11504,consumo_con_libranza,1095,82,193718.0,5100000.0,4966127.0,77007.0,42764,1,42764.0,...,600000.0,2,1,760.0,6,1,4,False,0,0.9973
3208,consumo_con_libranza,1095,4726,137004.0,4020000.0,3729726.0,0.0,1597773,1,7897805.0,...,1300000.0,2,1,795.0,7,2,4,False,0,0.9986
1391,consumo_sin_libranza,1096,7756,163221.0,5000000.0,4176874.0,59000.0,55001,1,55001.0,...,450000.0,3,0,727.0,4,2,4,True,1,0.8515
5008,consumo_sin_libranza,1096,4835,862743.0,25100000.0,19796604.0,47514.0,3293124,1,12051637.0,...,1500000.0,2,1,645.0,7,3,1,False,0,0.9811


## ➜ **Probabilidad Asignada** 

Ese score representa la probabilidad (confianza) que tiene el modelo de que una observación pertenezca a la clase predicha.

In [30]:
predicciones_internas[["default", "prediction_label", "prediction_score"]].head()

,default,prediction_label,prediction_score
981,False,0,0.9489
11504,False,0,0.9973
3208,False,0,0.9986
1391,True,1,0.8515
5008,False,0,0.9811


Los valores de `prediction_score` muestran que el modelo LightGBM realiza predicciones con alta confianza, ya que la mayoría de las probabilidades están cercanas a 1. 

Esto indica que el modelo es muy seguro al clasificar las observaciones en su respectiva. Sin embargo, esta alta confianza debe interpretarse junto con las métricas de desempeño, ya que una alta seguridad no siempre garantiza que todas las predicciones sean correctas.


El modelo está muy seguro en sus predicciones, pero esa seguridad no siempre significa que siempre acierta.



## ➜ **Parámetros del mejor modelo** 

In [33]:
lightgbm_tuned.get_params()

{'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 1.0,
 'importance_type': 'split',
 'learning_rate': 0.2,
 'max_depth': -1,
 'min_child_samples': 6,
 'min_child_weight': 0.001,
 'min_split_gain': 0.6,
 'n_estimators': 100,
 'n_jobs': -1,
 'num_leaves': 30,
 'objective': None,
 'random_state': 42,
 'reg_alpha': 0.001,
 'reg_lambda': 5,
 'subsample': 1.0,
 'subsample_for_bin': 200000,
 'subsample_freq': 0,
 'feature_fraction': 0.8,
 'bagging_freq': 3,
 'bagging_fraction': 0.8}

## ➜ **Modelo en datos no vistos**

Aquí se van a hacer pruebas al modelo con datos nunca vistos, vamos a simular nuevo clientes y vamos a verificar si generaliza fuera de los datos originales

In [37]:
np.random.seed(57)

n_samples = 50

df_synthetic_test = pd.DataFrame({
    "cartera": np.random.randint(0, 3, size=n_samples),
    "plazo": np.random.randint(1, 60, size=n_samples),
    "vinculacion": np.random.randint(60,30000 , size=n_samples),
    "valor_cuota": np.random.uniform(10000, 200000, size=n_samples),
    "valor_prestamo": np.random.uniform(100000, 500000, size=n_samples),
    "saldo_capital": np.random.uniform(0, 500000, size=n_samples),
    "saldo_interes": np.random.uniform(0, 500000, size=n_samples),
    "aportes": np.random.uniform(0, 10000000, size=n_samples),
    "garantias": np.random.randint(1, 9, size=n_samples),
    "valorgarantia": np.random.uniform(0, 500000, size=n_samples),
    "ctasahorros": np.random.randint(0, 10, size=n_samples),
    "edad": np.random.randint(18, 90, size=n_samples),
    "tipoasociado": np.random.randint(0, 1, size=n_samples),
    "sexo": np.random.randint(0, 1, size=n_samples),
    "curtotalingresos": np.random.uniform(500000, 1000000, size=n_samples),
    "curtotalegresos": np.random.uniform(100000, 800000, size=n_samples),
    "intestrato": np.random.randint(1, 6, size=n_samples),
    "actualizacion": np.random.randint(0, 1, size=n_samples),
    "puntaje_data": np.random.uniform(300, 850, size=n_samples),
    "grupo_ciudad": np.random.randint(1, 7, size=n_samples),
    "grupo_edad": np.random.randint(1, 5, size=n_samples),
    "grupo_actividadeco": np.random.randint(1, 4, size=n_samples)
})

df_synthetic_test.head()

,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,tipoasociado,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,puntaje_data,grupo_ciudad,grupo_edad,grupo_actividadeco
0,2,11,24177,59297.145066,276349.242057,20427.542503,224062.785917,1.038406e+06,1,255040.803804,...,0,0,823526.518924,197942.117402,4,0,682.836460,6,4,3
1,1,41,3316,33419.008452,315223.358449,113111.702882,181438.737036,2.398568e+05,8,282529.070925,...,0,0,634571.751615,343721.069687,2,0,708.962039,2,2,2
2,2,57,14136,47388.098216,140797.833081,15248.147669,490218.422936,4.144600e+06,7,6574.763111,...,0,0,991975.075638,584289.316111,4,0,325.700244,1,3,3
3,0,53,5906,172163.246463,377679.442188,457167.006453,372059.571489,9.228303e+06,3,439244.144870,...,0,0,681351.022210,452465.227374,4,0,649.301119,5,1,1
4,2,47,13544,68161.049122,206794.961126,86284.866578,139955.643283,8.634509e+06,1,202261.850510,...,0,0,755799.585589,107181.857010,3,0,351.736859,4,4,2


In [39]:
# Aplicar el modelo ajustado a los datos sintéticos de prueba
predicciones_sinteticas = predict_model(lightgbm_tuned, data=df_synthetic_test)

predicciones_sinteticas.head()

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


,cartera,plazo,vinculacion,valor_cuota,valor_prestamo,saldo_capital,saldo_interes,aportes,garantias,valorgarantia,...,curtotalingresos,curtotalegresos,intestrato,actualizacion,puntaje_data,grupo_ciudad,grupo_edad,grupo_actividadeco,prediction_label,prediction_score
0,2,11,24177,59297.144531,276349.250000,20427.542969,224062.781250,1.038406e+06,1,255040.796875,...,823526.5000,197942.125000,4,0,682.836487,6,4,3,1,0.9740
1,1,41,3316,33419.007812,315223.343750,113111.703125,181438.734375,2.398568e+05,8,282529.062500,...,634571.7500,343721.062500,2,0,708.962036,2,2,2,1,0.9850
2,2,57,14136,47388.097656,140797.828125,15248.147461,490218.437500,4.144600e+06,7,6574.763184,...,991975.0625,584289.312500,4,0,325.700256,1,3,3,1,0.9778
3,0,53,5906,172163.250000,377679.437500,457167.000000,372059.562500,9.228303e+06,3,439244.156250,...,681351.0000,452465.218750,4,0,649.301147,5,1,1,1,0.9289
4,2,47,13544,68161.046875,206794.968750,86284.867188,139955.640625,8.634509e+06,1,202261.843750,...,755799.5625,107181.859375,3,0,351.736847,4,4,2,1,0.9561


Los resultados obtenidos con datos sintéticos en el modelo **LightGBM** muestran predicciones con alta confianza, reflejada en prediction_score superiores al 90%.

Sin embargo, estos resultados deben interpretarse únicamente como una validación funcional del modelo, ya que los datos son simulados y no representan completamente la distribución real del problema.

## ➜ **Guardar el Modelo**

Indicamos la ruta donde queremos que se guarde nuestro modelo.

In [42]:
MODELS_DIR = Path.cwd().parent / "models"
MODELS_DIR.mkdir(exist_ok=True)

In [43]:
model_path = MODELS_DIR / 'lightgbm_tuned_model'
save_model(lightgbm_tuned, model_name=str(model_path))

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('onehot_encoding',
                  TransformerWrapper(exclude=None, include=['cartera'],
                                     transformer=OneHotEncoder(cols=['cartera'],
                                                               drop_invariant=False,
                                                               handle_missing='return_nan',
                                                               handle_unknown='value',
                                                               return_df=True,
                                                               use_cat_names=True,
                                                               verbose=0))),
                 ('trained_model',
                  LGBMClassifier(bagging_fraction=0.8, bagging_freq=3,
                                 boos...lass_weight=None,
                                 colsample_bytree=1.0, feature_fraction=0.8,
                          

## ➜ **Cargar el Modelo**

In [ ]:
lightgbm_tuned_loaded = load_model(str(MODELS_DIR /'lightgbm_tuned_model'))

Transformation Pipeline and Model Successfully Loaded


En este caso, el modelo se guardará también en formato **joblib**, ya que, según las necesidades del proyecto real y la decisión de los responsables, este es el formato elegido para la persistencia del modelo.


In [46]:
joblib.dump(lightgbm_tuned, MODELS_DIR / 'lightgbm_tuned_model.joblib')
lightgbm_tuned_loaded = joblib.load(MODELS_DIR / 'lightgbm_tuned_model.joblib')